<h1>Chapters 5 & 6 - Skills</h1>
<i>Adding specialized skills to your Agent that are used with ReACT.</i>


<a href="..."><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="..."><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="..."><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](...)

---

This notebook is for Chapters 5 and 6 of the [An Illustrated Guide to AI Agents](...) book by [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/) and [Jay Alammar](https://www.linkedin.com/in/jalammar).

---

<a href="...">
<img src="https://learning.oreilly.com/covers/urn:orm:book:9798341662681/400w/" width="350"/></a>


### **[OPTIONAL]** - Installing Packages on <img src="https://colab.google/static/images/icons/colab.png" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** one of the following codeblock to install the dependencies for this chapter. If you want to use a cloud provider, you only need to run the following code block:

In [ ]:
# %%capture
# !pip install illustrated-agents

---

💡 **NOTE**: If you want to use the GPU with `ollama`, then you will have to select a GPU first. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**. 

Then, **uncomment** and run this codeblock:

---

In [ ]:
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma3:12b &

# ▂▂▂▂▂▂▂▂▂▂▂▂

## 1 - Choosing Your LLM

At the beginning of every chapter, we start by choosing the LLM that we want to use:

In [ ]:
import os
from illustrated_agents.llm import LLM

# Ollama
llm = LLM(model="ollama/gemma3:12b")

# Llama.cpp server
# llm = LLM(model="openai/gemma-3-12b-it-Q4_K_M", api_base="http://localhost:8080", api_key="sk-no-key-required")

# Llama-cpp-python server
# llm = LLM(model="openai/gemma-3-12b-it-Q4_K_M.gguf", api_base="http://localhost:8000/v1/", api_key="sk-no-key-required")

# LM Studio
# llm = LLM(model="lm_studio/gemma-3-12b-it", api_base="http://localhost:1234/v1", api_key="sk-no-key-required")

# Google's Gemini / Gemma
# os.environ['GEMINI_API_KEY'] = "YOUR_GEMINI_API_KEY"
# llm = LLM(model="gemini/gemini-2.5-flash")
# llm = LLM(model="gemini/gemma-3-12b-it")

## 2 - Adding Recipes with **`SKILL.md`**

With modules like tools and MCP, we can give an Agent access to a number of tools or actions that it can take. However, when exactly to use those actions and how they fit into a larger workflow is not covered by any of these modules. This is where `SKILL.md` come in, they are a set of instructions on how to perform a given task. For instance, if you want it to create a presentation, you might need it to first search the web for relevant information (this is a tool), then summarize this information (it can do this by itself), and then finally create the slides one at a time (this is also a tool). This workflow or "recipe" for creating a presentation might therefore include a set of tools but also instructions on how to use them and in what order. As such,

> Skills teach your Agent what to do, when to do it, and how

The format of a `SKILL.md` file allows for proper context engineering. In the yaml frontmatter, there is the basic description of your skill which is always loaded into the context window:

```yaml
name: ...
description: ...
```

Below that, there is a more extensive description of the skill and how it should be executed. The full structure then becomes something like this:


```markdown
---
name: ...
description ...
---

#
...

##
...
```

This structure is especially helpful as you can write down extensive descriptions on how to use the skill, which may include domain-specific information. Skills are therefore especially helpful when you notice you have to repeat prompts often, like having to explain everytime the tone of voice that you want to or some domain-specific information regarding the schemas of your database.

Another benefit of skills is that they are **progressively disclosed**. This means that the yaml frontmatter is always given to the Agent as a system prompt, much like the tools we constructed in Chapter 5. However, the full markdown description is only given when the skill is **activated**. This therefore occupies a small amount of the context window and extends only when the skill is activated.

Next, let's explore how to give your `TinyAgent` access to these skills by first creating the `Skills` module:

In [ ]:
from pathlib import Path

import yaml


class Skills:
    """Skill loader for the Agent.

    Skills are recipes that teach the agent what to do, when to do it, and how.
    They are defined in SKILL.md files with YAML frontmatter (name, description)
    and markdown instructions.

    The skills are loaded progressively. As such, the name and description are
    available in the system prompt, but the full instructions are only injected
    when the agent **activates** a skill.
    """

    def __init__(self):
        self.skills = {}

    def load_from_file(self, path: str):
        """Load a skill from a SKILL.md file"""
        content = Path(path).read_text(encoding="utf-8")

        # Split on YAML delimiters and extract the frontmatter and body
        parts = content.split("---", 2)

        # Frontmatter
        frontmatter = yaml.safe_load(parts[1])
        name = frontmatter["name"]
        description = frontmatter["description"]

        # Body
        body = parts[2].strip()

        # Store the skill
        self.skills[name] = {
            "description": description,
            "instructions": body,
        }

    def activate(self, tool_call: dict) -> str:
        """Activate a skill and return formatted observation.

        Arguments:
            tool_call: A parsed tool call dict with "tool" and "args" keys.

        Returns:
            Formatted observation with skill instructions.
        """
        name = tool_call["args"][0]
        if name in self.skills:
            instructions = self.skills[name]["instructions"]
            return f"Skill '{name}' activated. Follow these instructions:\n\n{instructions}"
        return f"Skill '{name}' not found. Available skills: {', '.join(self.skills.keys())}"

    @property
    def prompt(self) -> str:
        """Generate prompt with skill descriptions (not full instructions)."""
        return f"""
# Skills

You have specialized skills available. To use a skill, call it like a tool:
{{"tool": "use_skill", "args": ["skill_name"]}}

The skill will provide detailed instructions for completing the task.

Available skills:
{self.descriptions}
"""

    @property
    def descriptions(self) -> str:
        """Get short descriptions of all skills."""
        return "\n".join(f"- `{name}`: {skill['description']}" for name, skill in self.skills.items())


Much like with the `Tools` module, there are only a couple of functions that we really need to add skills, starting with the prompt:

In [ ]:
from illustrated_agents.chapters.ch6_skills import skills_load_annotated; skills_load_annotated

In [ ]:
from illustrated_agents.chapters.ch6_skills import skills_class_annotated; skills_class_annotated


In [ ]:
from illustrated_agents.chapters.ch6_skills import skills_activate_annotated; skills_activate_annotated

Now that you have explored the code for loading and activating skills, let's explore how to actually create a skill. We already have prepared a skill for you to use, which can be found in `src/illustrated_agents/skills/file_analyzer`. The file contains all information about how to use our previously defined tool (`read_markdown`) along with a set of instructions on how to summarize its content. Let's load the skill and inspect it:

In [ ]:
import illustrated_agents

# We choose the file_analyzer skill as an example
file_analyzer_path = Path(illustrated_agents.__file__).parent / "skills" / "file_analyzer" / "SKILL.md"

# Load the skill
skills = Skills()
skills.load_from_file(file_analyzer_path)

Let's inspect the skill's description:

In [ ]:
print(skills.skills["file_analyzer"]["description"])

This is a short description of what the task is, but how to actually do it is covered in the full instruction:

In [ ]:
print(skills.skills["file_analyzer"]["instructions"])

Activating the skill would give back the full instruction as an observation:

In [ ]:
print(skills.activate({"tool": "use_skill", "args": ["file_analyzer"]}))

As you can see, the instruction is quite long and adding that to the system prompt would quickly fill up the context window if you have several skills that your Agent can use. 

Now, to implement the skill into your `TinyAgent`, we follow the same structure as we would with the tool usage since we are going to activate any given skill as if it were a tool. The changes to your `TinyAgent` are as follows:

In [ ]:
from illustrated_agents.chapters.ch6_skills import tinyagents_diff; tinyagents_diff

These changes result in your new `TinyAgent`:

In [ ]:
from illustrated_agents import LLM, Memory, Tools, MCPTools, ReAct, Reflector


class TinyAgent:
    """A minimal, modular, and educational agent framework."""

    def __init__(self, llm: LLM, memory: Memory, tools: Tools, planner: ReAct, reflector: Reflector, skills: Skills):
        self.llm = llm
        self.memory = memory
        self.tools = tools
        self.planner = planner
        self.reflector = reflector
        self.skills = skills

        # Build system prompt with all components
        system_prompt = "You are a helpful AI agent.\n\n"
        system_prompt += self.planner.prompt + "\n\n"
        system_prompt += self.skills.prompt
        system_prompt += self.tools.prompt
        self.memory.add("user", system_prompt)

    def run(self, task: str) -> str:
        """Run the agent on a task."""
        self.memory.add("user", task)

        # `Autonomy` loop
        for step in range(self.planner.max_steps):
            # Reflection step before taking the next action
            if self.reflector.should_reflect(step):
                self.memory.add("user", self.reflector.prompt)

            # Perform a step and check for completion
            result = self._step()
            if result is not None:
                return result

        return "Max steps reached without completion."

    def _step(self) -> str | None:
        """Perform a single step."""
        # Generate response and add to memory
        response = self.llm.generate(self.memory.get_messages())
        self.memory.add("assistant", response)

        # Parse planner's response to extract action if needed
        response = self.planner.parse(response)

        # Tool parsing and execution
        if self.tools.has_tool_call(response):
            return self._execute_action(response)

        return None

    def _execute_action(self, action: str) -> str | None:
        """Execute a tool action."""
        tool_call = self.tools.parse_tool_call(action)

        # Final answer ends the loop
        if tool_call["tool"] == "final_answer":
            return tool_call.get("args", "")

        # Activate skill and extract the observation
        if tool_call["tool"] == "use_skill":
            observation = self.skills.activate(tool_call)

        # Execute tool and extract the observation
        else:
            observation = self.tools.run_tool(tool_call)

        # Format the observation and add it to memory
        obs_prompt = f"OBSERVATION: {action} -> {observation}"
        self.memory.add("user", obs_prompt)

        return None

Finally, let's see if your newly updated `TinyAgent` will properly summarize any given markdown by using the skill and not only the tool:

In [ ]:
# Use MCP (which has the `read_markdown` tool)
tools = MCPTools()

# Memory
memory = Memory()

# ReAct
react = ReAct(max_steps=10)

# Reflector
reflector = Reflector(interval=5)

# Skills
skills = Skills()
skills.load_from_file(file_analyzer_path)

# Create agent
agent = TinyAgent(llm=llm, tools=tools, memory=memory, planner=react, reflector=reflector, skills=skills)

In [ ]:
query = "Analyze the file at https://raw.githubusercontent.com/MaartenGr/BERTopic/refs/heads/master/README.md. Use the the `file_analyzer` skill for context."

print(agent.run(query))

The output is exactly as instructed by our particular skill, great! Note that we nudged the Agent to use the `file_analyzer` skill. The reason for doing so is that the model can be a bit stubborn and think it does not need this skill. It also depends on which model you use. For the examples, we tend to use a smaller model which might have some difficulties understanding the nuances needed to call both skills and tools. It is advised to try out a bigger model instead, like `Gemini 3` which has a free tier you can use.

To check how it used the skill, let's see how it did by exploring it's memory:

In [ ]:
agent.memory.get_messages()

# ▂▂▂▂▂▂▂▂▂▂▂▂